In [2]:
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil

In [ ]:
# Read the CSV and extract column names as a list
cols = pd.read_csv("../columns.csv")["Column Names"].tolist()
cols_p2 = pd.read_csv("../p2_columns.csv")["Column Names"].tolist()

print(cols)  # Display the list of column names
print(cols_p2)  # Display the list of column names

xlsx_files = [f for f in os.listdir('../planning_data/Monthly Ped Reports') if f.endswith(('.xlsx', '.xlsm'))]
xlsx_files

xlsx_files_p2 = [f for f in os.listdir('../planning_data/Xovis_sensors/P2') if f.endswith(('.xlsx', '.xlsm'))]
xlsx_files_p2

['From', 'To Time', 'Z01_T4-ChurchSt_RevDoor_IN', 'Z01_T4-ChurchSt_RevDoor_OUT', 'Z01_T4-ChurchSt_SwingDoor_IN', 'Z01_T4-ChurchSt_SwingDoor_OUT', 'Z02_T4-LibertySt_EastEsc46_IN', 'Z02_T4-LibertySt_EastEsc46_OUT', 'Z02_T4-LibertySt_WestEsc45_IN', 'Z02_T4-LibertySt_WestEsc45_OUT', 'Z02_T4-LibertySt_Stair_IN', 'Z02_T4-LibertySt_Stair_OUT', 'Z02_T4-LibertySt_Doors_IN', 'Z02_T4-LibertySt_Doors_OUT', 'Z03_T4-C1-StairEsc_StairElev_IN', 'Z03_T4-C1-StairEsc_StairElev_OUT', 'Z03_T4-C1-StairEsc_EastEsc46_IN', 'Z03_T4-C1-StairEsc_EastEsc46_OUT', 'Z03_T4-C1-StairEsc_WestEsc45_IN', 'Z03_T4-C1-StairEsc_WestEsc45_OUT', 'Z04_T4-SoConc-Entry_AllDoors_IN', 'Z04_T4-SoConc-Entry_AllDoors_OUT', 'Z05_WestProj-Street_NorthDoors_IN', 'Z05_WestProj-Street_NorthDoors_OUT', 'Z05_WestProj-Street_SouthDoors_IN', 'Z05_WestProj-Street_SouthDoors_OUT', 'Z06_WestProj-C1-StairEsc_NorthStair_IN', 'Z06_WestProj-C1-StairEsc_NorthStair_OUT', 'Z06_WestProj-C1-StairEsc_NorthEsc36_IN', 'Z06_WestProj-C1-StairEsc_NorthEsc36_OUT'

[]

In [8]:
for file in xlsx_files:
    file_path = os.path.join('../planning_data/Monthly Ped Reports', file)
    arch_path = os.path.join('../planning_data/Monthly Ped Reports/processed', file)

    try:
        # Read the Excel file with specified columns
        df = pd.read_excel(file_path, sheet_name="Data", usecols=cols, engine="openpyxl")
        df = df[df["From"].notnull()]
        print(f"Successfully read: {file}")
        # Generate the parquet filename dynamically
        parquet_filename = f"../planning_data/ped_report_parquet/{file.rsplit('.', 1)[0]}.parquet"
        # Save as Parquet
        df.to_parquet(parquet_filename, engine="pyarrow", index=False)
        print(f"Parquet file saved: {parquet_filename}")
        # Move the original file to archive
        dest = shutil.move(file_path, arch_path) 

    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")


Successfully read: 2025-02_WTC-PED-MONTHLY-RPT.xlsm
Parquet file saved: ../planning_data/ped_report_parquet/2025-02_WTC-PED-MONTHLY-RPT.parquet
Successfully read: 2025-03_WTC-PED-MONTHLY-RPT.xlsm
Parquet file saved: ../planning_data/ped_report_parquet/2025-03_WTC-PED-MONTHLY-RPT.parquet
No valid data to save.


In [ ]:
possible_sheets = ["5-Min Data", "Sheet1", "Data"]  # Add all possible sheet names here

for file in xlsx_files:
    file_path = os.path.join('../planning_data/Xovis_sensors/P2', file)
    arch_path = os.path.join('../planning_data/Xovis_sensors/processed', file)

    df = None
    for sheet in possible_sheets:
        try:
            df = pd.read_excel(file_path, sheet_name=sheet, usecols=cols, engine="openpyxl")
            if df["From"].notnull().any():
                print(f"Successfully read '{file}' using sheet '{sheet}'")
                break  # Exit the loop if successfully read
        except Exception as e:
            continue  # Try next sheet

    if df is None or df["From"].notnull().sum() == 0:
        print(f"No valid data found in {file}")
        continue

    try:
        df = df[df["From"].notnull()]
        parquet_filename = f"../planning_data/p2_sensors_parquet/{file.rsplit('.', 1)[0]}.parquet"
        df.to_parquet(parquet_filename, engine="pyarrow", index=False)
        print(f"Parquet file saved: {parquet_filename}")
        shutil.move(file_path, arch_path)
    except Exception as e:
        print(f"Error processing {file}: {e}")
